# MNIST CNN with Random Test Visualization

This notebook trains a CNN on MNIST, plots training curves, evaluates the model, and then randomly tests 10 images from the test set with a matplotlib grid.

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import mnist
from tensorflow.keras.layers import Dense, Conv2D, MaxPool2D, Input, Flatten, Dropout, BatchNormalization
from tensorflow.keras.models import Model

np.random.seed(42)
tf.random.set_seed(42)

## Load MNIST dataset

In [ ]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()

## Preprocess data (normalize and reshape)

In [ ]:
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

x_train = np.expand_dims(x_train, axis=-1)
x_test = np.expand_dims(x_test, axis=-1)

y_train = y_train.astype("int64")
y_test = y_test.astype("int64")

## Build CNN model architecture

In [ ]:
inputs = Input((28, 28, 1))

x = Conv2D(32, 3, padding="same", activation="relu")(inputs)
x = BatchNormalization()(x)
x = Conv2D(32, 3, activation="relu")(x)
x = MaxPool2D()(x)
x = Dropout(0.25)(x)

x = Conv2D(64, 3, padding="same", activation="relu")(x)
x = BatchNormalization()(x)
x = Conv2D(64, 3, activation="relu")(x)
x = MaxPool2D()(x)
x = Dropout(0.25)(x)

x = Conv2D(128, 3, padding="same", activation="relu")(x)
x = BatchNormalization()(x)
x = MaxPool2D()(x)
x = Dropout(0.30)(x)

x = Flatten()(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.50)(x)
outputs = Dense(10, activation="softmax")(x)

model = Model(inputs, outputs)
model.summary()

## Compile model and configure callbacks

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

es = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True,
)

## Train model with validation split

In [ ]:
history = model.fit(
    x_train,
    y_train,
    epochs=100,
    validation_split=0.1,
    callbacks=[es],
)

## Plot training and validation loss and accuracy

In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history["loss"], label="loss")
plt.plot(history.history["val_loss"], label="val_loss")
plt.title("Loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["accuracy"], label="accuracy")
plt.plot(history.history["val_accuracy"], label="val_accuracy")
plt.title("Accuracy")
plt.legend()
plt.tight_layout()
plt.show()

## Evaluate model on train and test sets

In [ ]:
train_loss, train_acc = model.evaluate(x_train, y_train, verbose=0)
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)

print(f"Train loss: {train_loss:.4f}, Train accuracy: {train_acc:.4f}")
print(f"Test loss: {test_loss:.4f}, Test accuracy: {test_acc:.4f}")

## Randomly test 10 images and display predictions with plt

In [ ]:
indices = np.random.choice(len(x_test), size=10, replace=False)
images = x_test[indices]
true_labels = y_test[indices]
pred_probs = model.predict(images, verbose=0)
pred_labels = np.argmax(pred_probs, axis=1)

plt.figure(figsize=(15, 6))
for i, idx in enumerate(indices):
    is_correct = pred_labels[i] == true_labels[i]
    title_color = "green" if is_correct else "red"
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_test[idx].squeeze(), cmap="gray")
    plt.axis("off")
    plt.title(
        f"T:{true_labels[i]} P:{pred_labels[i]}\n{'Correct' if is_correct else 'Wrong'}",
        color=title_color,
        fontsize=10,
    )

plt.tight_layout()
plt.show()

## Save trained model

In [ ]:
model.save("mnist_cnn_model.keras")